# **5G Deployment Example**

In this example, we will deploy a 5G network with:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OCUDU CU-CP, CU-UP, and DU with ZeroMQ
- SRSUE with ZeroMQ as simulated UE

Reference: https://docs.ocudu.org/tutorials/srsue/#zeromq-based-setup

First, we install the required dependencies:

## Install Dependencies:

Install the `ifran` library:

In [ ]:
! pip install ifran

Install the Ansible dependencies:

In [ ]:
! ansible-galaxy install -r ../../ansible/requirements.yaml

Install the SLICES CLI used for the BI to create VMs:

In [ ]:
! pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
! slices auth login

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

In [ ]:
from ifran import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  sim_ru_du:
    txGainDB: 0
    rxGainDB: 0

  sim_ru_ue:
    txGainDB: 50
    rxGainDB: 40

ran:
  components:
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
      config:
        cu_cp: cucp
    du:
      type: du
      config:
        ru: sim_ru_du
        cu_cp: cucp
        cu_up: cuup 
        radioConfiguration:
          nrBand: 3
          downlinkArfcn: 368500
          channelBandwidthMhz: 10
          subcarrierSpacingKhz: 15
          ssbPeriodicityMs: 10
          prachConfigurationIndex: 1

ues:
  sim_ue:
    config:
      nrBands: [3]
      ru: sim_ru_ue
      gnbs: [du]

slices:
  internet:
    sst: 1
    sd: 0xFFFFFF # explicitly set sd to avoid core regection
"""
)

In [ ]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    project: post5g-beta
    experiment: ifran
    flavor: xlarge # ocudu zmq needs more than 5 CPU cores (see: https://github.com/srsran/srsRAN_Project/issues/1472)
    extra_path: /Users/zimabrou/Projects/5g_blueprint/venv/bin

ran:
  implementation: ocudu
  components:
    cucp:
      type: cu_cp
      advanced_config:
        inactivity_timer: 7200
    du:
      type: du
      advanced_config:
        srate: 11.52
        base_srate: "11.52e6"
        pdcch:
          common:
            ss0_index: 0
            coreset0_index: 6
          dedicated:
            ss2_type: common
            dci_format_0_1_and_1_1: false
        pdsch:
          mcs_table: qam64
        pusch:
          mcs_table: qam64

ues:
  sim_ue:
    type: containerized
    implementation: srsue
    advanced_config:
      srate: "11.52e6"
"""
)

## Provisioning:

In [ ]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [6]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node2' provisioning] ***
changed: [localhost]

TASK [provisioning/bi/provision : Delete experiment with name 'ifran' if it exists before attempting create for 'node2'] ***
skipping: [localhost]

TASK [provisioning/bi/provision : Wait for experiment deletion to take effect for 'node2'] ***
skipping: [localhost]

TASK [provisioning/bi/provision : Try to create experiment 'ifran' for 'node2'] ***
skipping: [localhost]

TASK [provisioning/bi/provision : Define SLICES_BI_INFRA and SLICES_EXPERIMENT environment variables for 'node2'] ***
ok: [localhost]

TASK [provisioning/bi/provision : Create 'node2' on infrastructure 'fr-sophia2-bi-vm1' using expe

## Kubernetes Deployment:

In [ ]:
from ifran import K8SManagement

k8s = K8SManagement(prov)

In [8]:
await k8s.deploy()


PLAY [node1] *******************************************************************

PLAY [node2] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************

PLAY [node3] *******************************************************************

PLAY [node4] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node4'] ****************
[WARNING]: Host 'node2' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node2]

TASK [k8

## 5G Core Deployment:

In [ ]:
from ifran import CoreManagement

core = CoreManagement(k8s)

In [10]:
await core.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-pcf' chart] ******

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

TASK [helm/prepare : Create Helm build directory for 'open5gs-upf' chart] ******

PLAY [node1] 

## RAN Deployment:

In [ ]:
from ifran import RANManagement

ran = RANManagement(core)

In [43]:
await ran.deploy()


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'ocudu-cuup' Helm chart] **********************

TASK [helm/uninstall : Uninstall 'ocudu-cucp' Helm chart] **********************

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'ocudu-du' Helm chart] ************************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: Deprecation warnings can be disabled by setting `deprecation_warnings=False` in ansible.cfg.
[DEPRECATION WARNING]: The 'status.values' return value is deprecated and will be remo

Get OCUDU ZMQ DU logs:

In [32]:
print(await ran.get_logs(components=["du"], quiet=True))


--== OCUDU DU (commit 050a2bb) ==--

2026-10-09T14:17:56.848239 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-10-09T14:17:56.849093 [CONFIG  ] [I] Input configuration (only non-default values): 
dryrun: false
gnb_id: 1
gnb_id_bit_length: 32
log:
  filename: stdout
  all_level: info
  config_level: info
metrics:
  autostart_stdout_metrics: true
f1ap:
  addrs: 10.244.160.19
  bind_addrs: 10.244.144.87
f1u:
  bind_port: 2153
  peer_port: 2153
  socket: "bind_addr: 10.244.144.87"
cell_cfg:
  pci: 0
  dl_arfcn: 368500
  band: 3
  common_scs: 15
  channel_bandwidth_MHz: 10
  nof_antennas_ul: 1
  nof_antennas_dl: 1
  plmn: 00101
  tac: 1
  slicing: "sst: 1\nsd: 16777215"
  ssb:
    ssb_period: 10
  pdcch:
    common:
      coreset0_index: 6
      ss0_index: 0
    dedicated:
      dci_format_0_1_and_1_1: false
      ss2_type: common
  pdsch:
    mcs_table: qam64
  pusch:
    mcs_table: qam64
  prach:
    prach_config_index: 1
ru_sdr:
  srate: 11.52
  device_dri

## UE Deployement:

In [ ]:
from ifran import UEManagement

ue = UEManagement(ran)

In [42]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'srsue-sim-ue' Helm chart] ********************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: Deprecation warnings can be disabled by setting `deprecation_warnings=False` in ansible.cfg.
[DEPRECATION WARNING]: The 'status.values' return value is deprecated and will be removed in version 8.0.0. Use 'status.release_values' instead. This feature will be removed from collection 'kubernetes.core' version 8.0.0.
changed: [node1]

TASK [helm/uninstall : Wait for 'srsue-sim-ue' pod to be deleted] **************
FAILED - RETRYING: [node1]: helm/uninstall : Wait for 'srsue-sim-ue' pod to be deleted (59 r

Get SRSUE ZMQ UE logs:

In [35]:
print(await ue.get_logs(components=["sim_ue"], quiet=True))

Active RF plugins: libsrsran_rf_uhd.so libsrsran_rf_zmq.so
Inactive RF plugins: 
Reading configuration file /etc/config/ue.conf...

Built in Release mode using commit 6bcbd9e on branch HEAD.

2026-10-09T14:18:38.472793 [UE     ] [I] Using binary /usr/local/bin/srsue with arguments: /etc/config/ue.conf 
2026-10-09T14:18:38.554721 [UE     ] [I] Built in Release mode using commit 6bcbd9e on branch HEAD.
2026-10-09T14:18:38.649473 [RF     ] [I] Configuring physical DL channel 0 with band-pass filter (0.0, 0.0)
2026-10-09T14:18:38.649474 [RF     ] [I] Configuring physical UL channel 0 with band-pass filter (0.0, 0.0)
Opening 1 channels in RF device=zmq with args=tx_port=tcp://10.244.82.222:2001,rx_port=tcp://10.244.144.87:2000,base_srate=11.52e6
Supported RF device list: UHD zmq file
CHx base_srate=11.52e6
Current sample rate is 1.92 MHz with a base rate of 11.52 MHz (x6 decimation)
CH0 rx_port=tcp://10.244.144.87:2000
CH0 tx_port=tcp://10.244.82.222:2001
Current sample rate is 11.52 MHz wi

Get DU logs after UE was connected:

In [36]:
print(await ran.get_logs(components=["du"], quiet=True))


--== OCUDU DU (commit 050a2bb) ==--

2026-10-09T14:17:56.848239 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-10-09T14:17:56.849093 [CONFIG  ] [I] Input configuration (only non-default values): 
dryrun: false
gnb_id: 1
gnb_id_bit_length: 32
log:
  filename: stdout
  all_level: info
  config_level: info
metrics:
  autostart_stdout_metrics: true
f1ap:
  addrs: 10.244.160.19
  bind_addrs: 10.244.144.87
f1u:
  bind_port: 2153
  peer_port: 2153
  socket: "bind_addr: 10.244.144.87"
cell_cfg:
  pci: 0
  dl_arfcn: 368500
  band: 3
  common_scs: 15
  channel_bandwidth_MHz: 10
  nof_antennas_ul: 1
  nof_antennas_dl: 1
  plmn: 00101
  tac: 1
  slicing: "sst: 1\nsd: 16777215"
  ssb:
    ssb_period: 10
  pdcch:
    common:
      coreset0_index: 6
      ss0_index: 0
    dedicated:
      dci_format_0_1_and_1_1: false
      ss2_type: common
  pdsch:
    mcs_table: qam64
  pusch:
    mcs_table: qam64
  prach:
    prach_config_index: 1
ru_sdr:
  srate: 11.52
  device_dri

Verify that the UE got a PDU session:

In [ ]:
print(await ue.run_command(command=["ip", "addr"], quiet=True))

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if18: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9166 qdisc noqueue state UP group default qlen 1000
    link/ether a2:48:36:37:2f:67 brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 10.244.82.222/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::a048:36ff:fe37:2f67/64 scope link 
       valid_lft forever preferred_lft forever
3: tun_srsue: <POINTOPOINT,MULTICAST,NOARP,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UNKNOWN group default qlen 500
    link/none 
    inet 11.1.1.3/24 scope global tun_srsue
       valid_lft forever preferred_lft forever



#### Test connectivity:

In [ ]:
print(await ue.run_command(command=["apt", "update"], quiet=True))

Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:2 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [61.4 kB]
Get:3 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1381 kB]
Get:4 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1546 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2069 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [331 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble/restricted amd64 Packages [117 kB]
Get:13 http://archive.ubuntu.com/ubuntu nob

In [ ]:
print(await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True))

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 22 not upgraded.
Need to get 44.6 kB of archives.
After this operation, 124 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 iputils-ping amd64 3:20240117-1ubuntu0.1 [44.6 kB]
Fetched 44.6 kB in 0s (400 kB/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 35148 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20240117-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20240117-1ubuntu0.1) ...
Setting up iputils-ping (3:20240117-1ubuntu0.1) ...



In [ ]:
print(await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 11.1.1.1 (11.1.1.1) from 11.1.1.3 tun_srsue: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=40.7 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=29.2 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=19.6 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=31.0 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=20.5 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=32.7 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5004ms
rtt min/avg/max/mdev = 19.570/28.948/40.723/7.271 ms



In [ ]:
print(await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 tun_srsue: 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=28.3 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=42.2 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=34.1 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=26.9 ms
64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=26.8 ms
64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=25.3 ms

--- 8.8.8.8 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5004ms
rtt min/avg/max/mdev = 25.323/30.608/42.228/5.901 ms



## Test another split: Change RAN split from E1-split to F1-split

We can change the OCUDU RAN's split from E1 to F1 while keeping the already deployed Open5GS core.

To do so, remove the deployed UE and RAN:

In [ ]:
await ue.remove(quiet=True)
await ran.remove(quiet=True)

Remove existing OCUDU E1s-split RAN components:

In [49]:
ran.remove_component("cuup")
ran.remove_component("cucp")
ran.remove_component("du")

Add CU that will replace the previous CU-CP and CU-UP:

In [ ]:
from metamodel_3gpp.models.ran import CUComponent, RANComponentType
from ifran.models.ran import DeployedRANComponent

ran.add_component(
    name="cu",
    component_meta=CUComponent(type=RANComponentType.CU),
    component_deployment=DeployedRANComponent.from_yaml(
"""
type: cu
implementation: ocudu
advanced_config:
  inactivity_timer: 7200
"""
    ),
)

Update and add the F1-split RAN DU from the previous one:

In [ ]:
du_meta = metamodel.ran.components["du"]
du_deployemnt = deployment_model.ran.components["du"]

# update cu_cp and cu_up references in du component
du_meta.config.cu_cp = "cu"
du_meta.config.cu_up = "cu"

ran.add_component(
    name="du",
    component_meta=du_meta,
    component_deployment=du_deployemnt,
)

Deploy the OCUDU F1-split RAN:

In [53]:
await ran.deploy()


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'ocudu-cu' chart] *********

TASK [helm/prepare : Create Helm build directory for 'ocudu-du' chart] *********
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/prepare : Copy local 'ocudu-du' Helm chart directory for 'ocudu-du'] ***
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference

Redeploy the SRSUE based on the changed RAN:

In [54]:
ue = UEManagement(ran)

In [55]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'srsue-sim-ue' chart] *****
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/prepare : Copy local 'srsue-containerized' Helm chart directory for 'srsue-sim-ue'] ***
ok: [node1]

TASK [helm/prepare : Render 'srsue-sim-ue' Helm Chart.yaml file from Jinja template] ***
ok: [node1]

TASK [helm/prepare : Render templated Helm files for 'srsue-sim-ue'] ***********
changed: [node1] => (item=values.yaml)
changed: [node1] => (item=config.yaml)

PLAY RECAP *********************************************************************
node1                      : ok=4    changed=1    

Get DU logs after UE was connected:

In [57]:
print(await ran.get_logs(components=["du"], quiet=True))


--== OCUDU DU (commit 050a2bb) ==--

2026-10-09T14:49:05.942867 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-10-09T14:49:05.943626 [CONFIG  ] [I] Input configuration (only non-default values): 
dryrun: false
gnb_id: 1
gnb_id_bit_length: 32
log:
  filename: stdout
  all_level: info
  config_level: info
metrics:
  autostart_stdout_metrics: true
f1ap:
  addrs: 10.244.111.240
  bind_addrs: 10.244.61.99
f1u:
  bind_port: 2153
  peer_port: 2153
  socket: "bind_addr: 10.244.61.99"
cell_cfg:
  pci: 0
  dl_arfcn: 368500
  band: 3
  common_scs: 15
  channel_bandwidth_MHz: 10
  nof_antennas_ul: 1
  nof_antennas_dl: 1
  plmn: 00101
  tac: 1
  slicing: "sst: 1\nsd: 16777215"
  ssb:
    ssb_period: 10
  pdcch:
    common:
      coreset0_index: 6
      ss0_index: 0
    dedicated:
      dci_format_0_1_and_1_1: false
      ss2_type: common
  pdsch:
    mcs_table: qam64
  pusch:
    mcs_table: qam64
  prach:
    prach_config_index: 1
ru_sdr:
  srate: 11.52
  device_driv

Verify that the UE got a PDU session:

In [ ]:
print(await ue.run_command(command=["ip", "addr"], quiet=True))

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if21: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9166 qdisc noqueue state UP group default qlen 1000
    link/ether 8e:8f:19:c9:88:55 brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 10.244.204.182/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::8c8f:19ff:fec9:8855/64 scope link 
       valid_lft forever preferred_lft forever
3: tun_srsue: <POINTOPOINT,MULTICAST,NOARP,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UNKNOWN group default qlen 500
    link/none 
    inet 11.1.1.4/24 scope global tun_srsue
       valid_lft forever preferred_lft forever



#### Test connectivity:

In [ ]:
print(await ue.run_command(command=["apt", "update"], quiet=True))

Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:2 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [61.4 kB]
Get:3 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1381 kB]
Get:4 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2069 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1546 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble/restricted amd64 Packages [117 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [331 kB]
Get:13 http://archive.ubuntu.com/ubuntu nob

In [ ]:
print(await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True)

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 22 not upgraded.
Need to get 44.6 kB of archives.
After this operation, 124 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 iputils-ping amd64 3:20240117-1ubuntu0.1 [44.6 kB]
Fetched 44.6 kB in 30s (1475 B/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 35148 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20240117-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20240117-1ubuntu0.1) ...
Setting up iputils-ping (3:20240117-1ubuntu0.1) ...



In [ ]:
print(await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 11.1.1.1 (11.1.1.1) from 11.1.1.4 tun_srsue: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=46.3 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=35.0 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=49.1 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=38.9 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=63.8 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=20.2 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5004ms
rtt min/avg/max/mdev = 20.181/42.224/63.773/13.398 ms



In [ ]:
print(await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 8.8.8.8 (8.8.8.8) from 11.1.1.4 tun_srsue: 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=39.5 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=29.1 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=25.2 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=42.7 ms
64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=38.4 ms
64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=34.0 ms

--- 8.8.8.8 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5004ms
rtt min/avg/max/mdev = 25.223/34.818/42.689/6.072 ms

